In [3]:
from google.colab import files

uploaded = files.upload()


Saving opportunities.csv to opportunities.csv
Saving facilities.csv to facilities.csv


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import radians, sin, cos, asin, sqrt
from copy import deepcopy
from scipy.stats import variation

pd.set_option("display.max_columns", None)

dist_weight = 0.4

complexity_weight = 0.4

cap_buffer = 1.1

sl_penalty = 0.2

reassignment_penalty = 0.3

overload_weight = 0.5


In [5]:
def haversine_miles(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return 3959 * c


def scale(value, min_val, max_val):
    if pd.isna(value) or pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return 0.0
    return (value - min_val) / (max_val - min_val)


In [6]:
facilities = pd.read_csv("facilities.csv")
opportunities = pd.read_csv("opportunities.csv")

facilities.columns = facilities.columns.str.strip()
opportunities.columns = opportunities.columns.str.strip()

facilities["VP ID"] = facilities["VP ID"].astype(str).str.strip()
facilities["Facility ID"] = facilities["Facility ID"].astype(str).str.strip()

if "Facility ID" in opportunities.columns:
    opportunities["Facility ID"] = opportunities["Facility ID"].astype(str).str.strip()

for col in ["Latitude", "Longitude", "Facility Complexity Score"]:
    if col in facilities.columns:
        facilities[col] = pd.to_numeric(facilities[col], errors="coerce")
    if col in opportunities.columns:
        opportunities[col] = pd.to_numeric(opportunities[col], errors="coerce")

facilities["record_type"] = "current"
opportunities["record_type"] = "opportunity"

if "Facility ID" not in opportunities.columns:
    opportunities["Facility ID"] = ["OPP_" + str(i) for i in range(len(opportunities))]
else:
    opportunities["Facility ID"] = opportunities["Facility ID"].fillna(
        pd.Series(["OPP_" + str(i) for i in range(len(opportunities))], index=opportunities.index)
    )

if "Service Line" not in facilities.columns:
    facilities["Service Line"] = "Unknown"

if "Service Line" not in opportunities.columns:
    opportunities["Service Line"] = "Unknown"


In [7]:
req_fac_cols = [
    "Facility ID", "Latitude", "Longitude",
    "Facility Complexity Score", "Service Line",
    "record_type", "VP ID"
]

req_opp_cols = [
    "Facility ID", "Latitude", "Longitude",
    "Facility Complexity Score", "Service Line",
    "record_type"
]

missing_fac = [col for col in req_fac_cols if col not in facilities.columns]

missing_opp = [col for col in req_opp_cols if col not in opportunities.columns]

if missing_fac:
    raise ValueError(f"Missing in facilities: {missing_fac}")

if missing_opp:
    raise ValueError(f"Missing in opportunities: {missing_opp}")

current_sol = dict(zip(facilities["Facility ID"], facilities["VP ID"]))

vp_centroids = (
    facilities.groupby("VP ID")
    .agg(
        centroid_lat=("Latitude", "mean"),
        centroid_lon=("Longitude", "mean")
    )
    .reset_index()
)

facilities = facilities.drop(
    columns=[col for col in ["centroid_lat", "centroid_lon", "centroid_lat_x", "centroid_lat_y", "centroid_lon_x", "centroid_lon_y"] if col in facilities.columns],
    errors="ignore"
)


facilities = facilities.merge(vp_centroids, on="VP ID", how="left")

facilities["miles_to_centroid"] = facilities.apply(
    lambda row: haversine_miles(
        row["Latitude"],
        row["Longitude"],
        row["centroid_lat"],
        row["centroid_lon"]
    ),
    axis=1
)

all_entities = pd.concat(
    [
        facilities[req_fac_cols].rename(columns={"VP ID": "current_vp"}),
        opportunities[req_opp_cols].assign(current_vp=np.nan)
    ],
    ignore_index=True
)

all_entities.head()


,Facility ID,Latitude,Longitude,Facility Complexity Score,Service Line,record_type,current_vp
0,17,27.81,-82.73,6.043751,EVS,current,VP0050
1,40,30.09,-94.13,23.951923,EVS,current,VP0028
2,67,30.40,-91.10,17.637183,EVS,current,VP0059
3,85,30.18,-82.69,6.013388,EVS,current,VP0041
4,117,30.91,-94.01,3.045059,EVS,current,VP0028


In [8]:
dist_min = facilities["miles_to_centroid"].min()

dist_max = facilities["miles_to_centroid"].max()

comp_min = facilities["Facility Complexity Score"].min()
comp_max = facilities["Facility Complexity Score"].max()

facilities["dist_score"] = facilities["miles_to_centroid"].apply(lambda x: scale(x, dist_min, dist_max))
facilities["complexity_score"] = facilities["Facility Complexity Score"].apply(lambda x: scale(x, comp_min, comp_max))

facilities["facility_workload"] = (
    dist_weight * facilities["dist_score"] +
    complexity_weight * facilities["complexity_score"]
)

vp_summary = (
    facilities.groupby("VP ID")
    .agg(
        facility_count=("Facility ID", "nunique"),
        sl_count=("Service Line", "nunique"),
        total_miles=("miles_to_centroid", "sum"),
        total_complexity=("Facility Complexity Score", "sum"),
        base_workload=("facility_workload", "sum")
    )
    .reset_index()
)

vp_sls = (
    facilities.groupby("VP ID")["Service Line"]
    .apply(lambda x: set(x.dropna()))
    .to_dict()
)

avg_workload = vp_summary["base_workload"].mean()
uniform_capacity = cap_buffer * avg_workload

vp_summary["capacity_threshold"] = uniform_capacity
vp_summary["remaining_capacity"] = vp_summary["capacity_threshold"] - vp_summary["base_workload"]
vp_summary["travel_cost_proxy"] = vp_summary["total_miles"]

sl_summary = (
    facilities.groupby(["VP ID", "Service Line"])
    .agg(
        facility_count=("Facility ID", "nunique"),
        total_workload=("facility_workload", "sum")
    )
    .reset_index()
)

vp_summary.head()


,VP ID,facility_count,sl_count,total_miles,total_complexity,base_workload,capacity_threshold,remaining_capacity,travel_cost_proxy
0,VP0001,1,1,0.000000,0.000000,0.000440,0.99507,0.994630,0.000000
1,VP0002,14,1,5896.511510,-4.335105,1.651319,0.99507,-0.656249,5896.511510
2,VP0003,1,1,0.000000,-0.318308,0.000000,0.99507,0.995070,0.000000
3,VP0004,18,1,6453.653784,1318.147997,3.638024,0.99507,-2.642954,6453.653784
4,VP0005,18,1,1456.617730,224.459978,0.726226,0.99507,0.268844,1456.617730


In [9]:
missing = facilities.isnull().sum()
missing = missing[missing > 0]


In [10]:
def compute_assignment_cost(entity, vp_row, vp_sls, current_sol):
    vp_id = vp_row['VP ID']
    d = haversine_miles(entity['Latitude'], entity['Longitude'], vp_row['centroid_lat'], vp_row['centroid_lon'])
    norm_d = scale(d, 0, 3000)
    norm_c = scale(entity['Facility Complexity Score'], comp_min, comp_max)
    entity_service = entity['Service Line']
    service_pen = 0 if vp_id in vp_sls and entity_service in vp_sls[vp_id] else 1
    reassignment_pen = 0
    if entity['record_type'] == 'current':
        current_vp = current_sol.get(entity['Facility ID'])
        if current_vp is not None and current_vp != vp_id:
            reassignment_pen = 1
    cost = dist_weight * norm_d + complexity_weight * norm_c + sl_penalty * service_pen + reassignment_penalty * reassignment_pen
    return 0 if pd.isna(cost) else cost
assignment_costs = {}
for _, entity in all_entities.iterrows():
    entity_id = entity['Facility ID']
    assignment_costs[entity_id] = {}
    for _, vp in vp_centroids.iterrows():
        vp_id = vp['VP ID']
        assignment_costs[entity_id][vp_id] = compute_assignment_cost(entity, vp, vp_sls, current_sol)
first_entity = list(assignment_costs.keys())[0]


In [11]:
vp_capacity = dict(zip(vp_summary["VP ID"], vp_summary["capacity_threshold"]))

vp_base_workload = dict(zip(vp_summary["VP ID"], vp_summary["base_workload"]))


def build_system_sol(all_entities, assignment_costs, vp_capacity, vp_base_workload):
    sol = {}
    current_workload = deepcopy(vp_base_workload)

    for _, entity in all_entities.iterrows():
        entity_id = entity["Facility ID"]

        feasible = []

        for vp_id, cost in assignment_costs[entity_id].items():
            if current_workload.get(vp_id, 0) + cost <= vp_capacity.get(vp_id, 0):
                feasible.append((vp_id, cost))

        if feasible:
            best_vp = min(feasible, key=lambda x: x[1])[0]
        else:
            best_vp = min(assignment_costs[entity_id].items(), key=lambda x: x[1])[0]

        sol[entity_id] = best_vp
        current_workload[best_vp] = current_workload.get(best_vp, 0) + assignment_costs[entity_id][best_vp]

    return sol, current_workload


def evaluate_system(sol, assignment_costs, vp_capacity, vp_base_workload):
    current_workload = deepcopy(vp_base_workload)
    total_cost = 0

    for entity_id, vp_id in sol.items():
        if entity_id not in assignment_costs or vp_id not in assignment_costs[entity_id]:
            continue

        cost = assignment_costs[entity_id][vp_id]
        cost = 0 if pd.isna(cost) else cost
        current_workload[vp_id] = current_workload.get(vp_id, 0) + cost
        total_cost += cost

    overload_total = 0
    for vp_id in current_workload:
        overload = max(0, current_workload[vp_id] - vp_capacity.get(vp_id, 0))
        overload_total += overload

    objective = total_cost + overload_weight * overload_total

    return objective, current_workload


def count_violations(workload, capacity):
    return sum(1 for vp_id in workload if workload[vp_id] > capacity.get(vp_id, 0))


In [12]:
candidate_sol, candidate_workload = build_system_sol(all_entities, assignment_costs, vp_capacity, vp_base_workload)

candidate_score, updated_workload = evaluate_system(candidate_sol, assignment_costs, vp_capacity, vp_base_workload)

overloaded = {vp: updated_workload[vp] for vp in updated_workload if updated_workload[vp] > vp_capacity.get(vp, 0)}

candidate_reassignments = sum((1 for fid, vp in candidate_sol.items() if fid in current_sol and current_sol[fid] != vp))


In [13]:
base_sol = current_sol.copy()

base_score, base_workload = evaluate_system(base_sol, assignment_costs, vp_capacity, vp_base_workload)

objective_delta = candidate_score - base_score

percent_improvement = (base_score - candidate_score) / base_score * 100 if base_score != 0 else 0

base_violations = count_violations(base_workload, vp_capacity)

candidate_violations = count_violations(updated_workload, vp_capacity)


In [14]:
def local_search(initial_sol, assignment_costs, vp_capacity, vp_base_workload, max_iter=50):
    best_sol = deepcopy(initial_sol)
    best_score, _ = evaluate_system(best_sol, assignment_costs, vp_capacity, vp_base_workload)
    for _ in range(max_iter):
        improved = False
        for entity_id in list(best_sol.keys()):
            current_vp = best_sol[entity_id]
            for vp_id in vp_capacity.keys():
                if vp_id == current_vp:
                    continue
                trial_sol = deepcopy(best_sol)
                trial_sol[entity_id] = vp_id
                trial_score, _ = evaluate_system(trial_sol, assignment_costs, vp_capacity, vp_base_workload)
                if trial_score < best_score:
                    best_sol = trial_sol
                    best_score = trial_score
                    improved = True
        if not improved:
            break
    return (best_sol, best_score)

optimized_sol, optimized_score = local_search(candidate_sol, assignment_costs, vp_capacity, vp_base_workload, max_iter=30)

optimized_score, optimized_workload = evaluate_system(optimized_sol, assignment_costs, vp_capacity, vp_base_workload)

improvement = candidate_score - optimized_score

percent_improvement = improvement / candidate_score * 100 if candidate_score != 0 else 0


In [15]:
optimized_violations = count_violations(optimized_workload, vp_capacity)

workload_changes = {vp_id: optimized_workload.get(vp_id, 0) - base_workload.get(vp_id, 0) for vp_id in vp_base_workload}

reassignments = {entity_id: (current_sol[entity_id], new_vp) for entity_id, new_vp in optimized_sol.items() if entity_id in current_sol and current_sol[entity_id] != new_vp}

candidate_to_optimized_changes = {entity_id: (candidate_sol[entity_id], optimized_sol[entity_id]) for entity_id in candidate_sol if candidate_sol[entity_id] != optimized_sol[entity_id]}

pd.DataFrame.from_dict(workload_changes, orient='index', columns=['workload_change']).head()


,workload_change
VP0001,0.552547
VP0002,0.000000
VP0003,0.000000
VP0004,0.000000
VP0005,0.008205


In [16]:
def compare_objective(base_obj, new_obj):
    delta = new_obj - base_obj
    percent_change = ((base_obj - new_obj) / base_obj) * 100 if base_obj != 0 else 0

    return {
        "base_objective": base_obj,
        "new_objective": new_obj,
        "objective_delta": delta,
        "percent_improvement": percent_change
    }


def compare_violations(base_violations, new_violations):
    return {
        "base_violations": base_violations,
        "new_violations": new_violations,
        "violation_delta": new_violations - base_violations
    }


def compute_workload_changes(base_workload, new_workload):
    return {
        vp_id: new_workload.get(vp_id, 0) - base_workload.get(vp_id, 0)
        for vp_id in base_workload
    }


def compute_reassignments(base_sol, new_sol):
    return {
        entity_id: (base_sol[entity_id], new_vp)
        for entity_id, new_vp in new_sol.items()
        if entity_id in base_sol and base_sol[entity_id] != new_vp
    }


def build_sl_map(sol, entity_df):
    service_map = {}
    entity_lookup_local = entity_df.drop_duplicates(subset=["Facility ID"]).set_index("Facility ID")

    for entity_id, vp_id in sol.items():
        if entity_id not in entity_lookup_local.index:
            continue

        service = entity_lookup_local.loc[entity_id].get("Service Line", "Unknown")
        service_map.setdefault(vp_id, set()).add(service)

    return service_map


def compare_sls(base_sls, new_sls):
    sl_changes = {}
    all_vps = set(base_sls.keys()).union(set(new_sls.keys()))

    for vp_id in all_vps:
        base_lines = base_sls.get(vp_id, set())
        new_lines = new_sls.get(vp_id, set())

        sl_changes[vp_id] = {
            "base_count": len(base_lines),
            "new_count": len(new_lines),
            "added_sls": list(new_lines - base_lines)
        }

    return sl_changes


def run_comparative_evaluation(
    base_sol,
    new_sol,
    base_obj,
    new_obj,
    base_workload,
    new_workload,
    base_violations,
    new_violations,
    base_sls,
    all_entities
):
    new_sls = build_sl_map(new_sol, all_entities)

    return {
        "objective_comparison": compare_objective(base_obj, new_obj),
        "violation_comparison": compare_violations(base_violations, new_violations),
        "workload_changes": compute_workload_changes(base_workload, new_workload),
        "reassignments": compute_reassignments(base_sol, new_sol),
        "sl_changes": compare_sls(base_sls, new_sls)
    }


In [17]:
comparison_results = run_comparative_evaluation(base_sol=base_sol, new_sol=optimized_sol, base_obj=base_score, new_obj=optimized_score, base_workload=base_workload, new_workload=optimized_workload, base_violations=base_violations, new_violations=optimized_violations, base_sls=vp_sls, all_entities=all_entities)
pd.DataFrame.from_dict(comparison_results['sl_changes'], orient='index').head()


,base_count,new_count,added_sls
VP0045,1,1,[]
VP0032,1,1,[]
VP0004,1,1,[]
VP0040,1,1,[]
VP0050,1,1,[]


In [18]:
def evaluate_system_with_overload_weight(sol, assignment_costs, vp_capacity, vp_base_workload, overload_weight_value):
    current_workload = deepcopy(vp_base_workload)
    total_cost = 0

    for entity_id, vp_id in sol.items():
        cost = assignment_costs.get(entity_id, {}).get(vp_id, 0)
        cost = 0 if pd.isna(cost) else cost
        current_workload[vp_id] = current_workload.get(vp_id, 0) + cost
        total_cost += cost

    overload_total = 0
    for vp_id in current_workload:
        overload = max(0, current_workload[vp_id] - vp_capacity.get(vp_id, 0))
        overload_total += overload

    objective = total_cost + overload_weight_value * overload_total

    return objective, current_workload


def local_search_with_overload_weight(initial_sol, assignment_costs, vp_capacity, vp_base_workload, overload_weight_value, max_iter=30):
    best_sol = deepcopy(initial_sol)
    best_score, _ = evaluate_system_with_overload_weight(
        best_sol,
        assignment_costs,
        vp_capacity,
        vp_base_workload,
        overload_weight_value
    )

    for _ in range(max_iter):
        improved = False

        for entity_id in list(best_sol.keys()):
            current_vp = best_sol[entity_id]

            for vp_id in vp_capacity.keys():
                if vp_id == current_vp:
                    continue

                trial_sol = deepcopy(best_sol)
                trial_sol[entity_id] = vp_id

                trial_score, _ = evaluate_system_with_overload_weight(
                    trial_sol,
                    assignment_costs,
                    vp_capacity,
                    vp_base_workload,
                    overload_weight_value
                )

                if trial_score < best_score:
                    best_sol = trial_sol
                    best_score = trial_score
                    improved = True

        if not improved:
            break

    return best_sol, best_score


def run_model_with_weights(
    all_entities,
    vp_centroids,
    vp_sls,
    current_sol,
    vp_capacity,
    vp_base_workload,
    distance_weight,
    complexity_weight,
    sl_penalty,
    reassignment_penalty,
    overload_weight_value
):
    def compute_assignment_cost_weighted(entity, vp_row):
        vp_id = vp_row["VP ID"]

        dist_cost = haversine_miles(
            entity["Latitude"],
            entity["Longitude"],
            vp_row["centroid_lat"],
            vp_row["centroid_lon"]
        )
        dist_cost = 0 if pd.isna(dist_cost) else dist_cost
        dist_score = scale(dist_cost, 0, 3000)

        raw_complexity = entity["Facility Complexity Score"]
        raw_complexity = comp_min if pd.isna(raw_complexity) else raw_complexity
        comp_score = scale(raw_complexity, comp_min, comp_max)
        comp_score = 0 if pd.isna(comp_score) else comp_score

        entity_service = entity["Service Line"]
        service_pen = 0 if vp_id in vp_sls and entity_service in vp_sls[vp_id] else 1

        reassignment_pen = 0
        if entity["record_type"] == "current":
            current_vp = current_sol.get(entity["Facility ID"])
            if current_vp is not None and current_vp != vp_id:
                reassignment_pen = 1

        cost = (
            distance_weight * dist_score +
            complexity_weight * comp_score +
            sl_penalty * service_pen +
            reassignment_penalty * reassignment_pen
        )

        return 0 if pd.isna(cost) else cost

    assignment_costs_test = {}

    for _, entity in all_entities.iterrows():
        entity_id = entity["Facility ID"]
        assignment_costs_test[entity_id] = {}

        for _, vp in vp_centroids.iterrows():
            vp_id = vp["VP ID"]
            assignment_costs_test[entity_id][vp_id] = compute_assignment_cost_weighted(entity, vp)

    candidate_sol_test, _ = build_system_sol(
        all_entities,
        assignment_costs_test,
        vp_capacity,
        vp_base_workload
    )

    candidate_score_test, _ = evaluate_system_with_overload_weight(
        candidate_sol_test,
        assignment_costs_test,
        vp_capacity,
        vp_base_workload,
        overload_weight_value
    )

    optimized_sol_test, optimized_score_test = local_search_with_overload_weight(
        candidate_sol_test,
        assignment_costs_test,
        vp_capacity,
        vp_base_workload,
        overload_weight_value,
        max_iter=30
    )

    optimized_score_test, optimized_workload_test = evaluate_system_with_overload_weight(
        optimized_sol_test,
        assignment_costs_test,
        vp_capacity,
        vp_base_workload,
        overload_weight_value
    )

    optimized_violations_test = count_violations(optimized_workload_test, vp_capacity)

    return {
        "candidate_score": candidate_score_test,
        "optimized_score": optimized_score_test,
        "violations": optimized_violations_test,
        "sol": optimized_sol_test,
        "workload": optimized_workload_test
    }


def run_sensitivity_analysis(weight_grid):
    results = []

    for config in weight_grid:
        result = run_model_with_weights(
            all_entities=all_entities,
            vp_centroids=vp_centroids,
            vp_sls=vp_sls,
            current_sol=current_sol,
            vp_capacity=vp_capacity,
            vp_base_workload=vp_base_workload,
            distance_weight=config["distance_weight"],
            complexity_weight=config["complexity_weight"],
            sl_penalty=config["sl_penalty"],
            reassignment_penalty=config["reassignment_penalty"],
            overload_weight_value=config["overload_weight"]
        )

        results.append({
            "distance_weight": config["distance_weight"],
            "complexity_weight": config["complexity_weight"],
            "sl_penalty": config["sl_penalty"],
            "reassignment_penalty": config["reassignment_penalty"],
            "overload_weight": config["overload_weight"],
            "candidate_score": result["candidate_score"],
            "optimized_score": result["optimized_score"],
            "violations": result["violations"]
        })

    return pd.DataFrame(results)


In [20]:
weight_configs = {
    "Balanced Growth": {
        "distance_weight": 0.4,
        "complexity_weight": 0.6,
        "sl_penalty": 0.2,
        "reassignment_penalty": 0.3,
        "overload_weight": 2.0
    },
    "Geographic Efficiency": {
        "distance_weight": 0.6,
        "complexity_weight": 0.4,
        "sl_penalty": 0.2,
        "reassignment_penalty": 0.3,
        "overload_weight": 2.0
    },
    "Service Line Fit": {
        "distance_weight": 0.5,
        "complexity_weight": 0.5,
        "sl_penalty": 0.4,
        "reassignment_penalty": 0.3,
        "overload_weight": 2.0
    },
    "Minimize Disruption": {
        "distance_weight": 0.5,
        "complexity_weight": 0.5,
        "sl_penalty": 0.2,
        "reassignment_penalty": 0.5,
        "overload_weight": 2.0
    }
}

weight_grid = list(weight_configs.values())

sensitivity_results = run_sensitivity_analysis(weight_grid)
sensitivity_results.insert(0, "scenario_name", list(weight_configs.keys()))
sensitivity_results


KeyboardInterrupt: 

In [ ]:
selected_scenario = "Balanced Growth"

selected_config = weight_configs[selected_scenario]

selected_result = run_model_with_weights(
    all_entities=all_entities,
    vp_centroids=vp_centroids,
    vp_sls=vp_sls,
    current_sol=current_sol,
    vp_capacity=vp_capacity,
    vp_base_workload=vp_base_workload,
    distance_weight=selected_config["distance_weight"],
    complexity_weight=selected_config["complexity_weight"],
    sl_penalty=selected_config["sl_penalty"],
    reassignment_penalty=selected_config["reassignment_penalty"],
    overload_weight_value=selected_config["overload_weight"]
)

selected_sol = selected_result["sol"]
selected_workload = selected_result["workload"]
selected_score = selected_result["optimized_score"]
selected_violations = selected_result["violations"]

In [ ]:
scenario_a = "Balanced Growth"
scenario_b = "Minimize Disruption"

config_a = weight_configs[scenario_a]
config_b = weight_configs[scenario_b]

result_a = run_model_with_weights(
    all_entities=all_entities,
    vp_centroids=vp_centroids,
    vp_sls=vp_sls,
    current_sol=current_sol,
    vp_capacity=vp_capacity,
    vp_base_workload=vp_base_workload,
    distance_weight=config_a["distance_weight"],
    complexity_weight=config_a["complexity_weight"],
    sl_penalty=config_a["sl_penalty"],
    reassignment_penalty=config_a["reassignment_penalty"],
    overload_weight_value=config_a["overload_weight"]
)

result_b = run_model_with_weights(
    all_entities=all_entities,
    vp_centroids=vp_centroids,
    vp_sls=vp_sls,
    current_sol=current_sol,
    vp_capacity=vp_capacity,
    vp_base_workload=vp_base_workload,
    distance_weight=config_b["distance_weight"],
    complexity_weight=config_b["complexity_weight"],
    sl_penalty=config_b["sl_penalty"],
    reassignment_penalty=config_b["reassignment_penalty"],
    overload_weight_value=config_b["overload_weight"]
)

scenario_comparison = pd.DataFrame([
    {
        "Scenario": scenario_a,
        "Optimized Score": result_a["optimized_score"],
        "Capacity Violations": result_a["violations"]
    },
    {
        "Scenario": scenario_b,
        "Optimized Score": result_b["optimized_score"],
        "Capacity Violations": result_b["violations"]
    }
])

scenario_comparison

In [ ]:
selected_scenario = "Balanced Growth"

In [ ]:
def summarize_sensitivity_stability(sensitivity_results):
    objective_cv = variation(sensitivity_results["optimized_score"])
    violations_cv = variation(sensitivity_results["violations"])

    return pd.DataFrame([
        {
            "Metric": "Optimized Objective Score",
            "Mean": sensitivity_results["optimized_score"].mean(),
            "Standard Deviation": sensitivity_results["optimized_score"].std(),
            "Coefficient of Variation": objective_cv
        },
        {
            "Metric": "Capacity Violations",
            "Mean": sensitivity_results["violations"].mean(),
            "Standard Deviation": sensitivity_results["violations"].std(),
            "Coefficient of Variation": violations_cv
        }
    ])

sensitivity_stability = summarize_sensitivity_stability(sensitivity_results)
sensitivity_stability


In [ ]:
def build_decision_table(sol, assignment_costs):
    rows = []

    for entity_id, vp_id in sol.items():
        rows.append({
            "Facility ID": entity_id,
            "Assigned VP": vp_id,
            "Assignment Cost": assignment_costs.get(entity_id, {}).get(vp_id, np.nan)
        })

    return pd.DataFrame(rows)


def summarize_overload(workload, capacity):
    rows = []

    for vp_id in workload:
        leader_workload = workload.get(vp_id, 0)
        leader_capacity = capacity.get(vp_id, 0)
        excess = max(0, leader_workload - leader_capacity)

        rows.append({
            "VP ID": vp_id,
            "Workload": leader_workload,
            "Capacity": leader_capacity,
            "Over Capacity": leader_workload > leader_capacity,
            "Excess": excess
        })

    return pd.DataFrame(rows).sort_values(by="Excess", ascending=False)


def objective_decomposition(sol, assignment_costs, vp_capacity, vp_base_workload, overload_weight_value):
    current_workload = deepcopy(vp_base_workload)
    assignment_total = 0

    for entity_id, vp_id in sol.items():
        assignment_cost = assignment_costs.get(entity_id, {}).get(vp_id, 0)
        assignment_cost = 0 if pd.isna(assignment_cost) else assignment_cost
        assignment_total += assignment_cost
        current_workload[vp_id] = current_workload.get(vp_id, 0) + assignment_cost

    overload_total = 0

    for vp_id in current_workload:
        excess = max(0, current_workload[vp_id] - vp_capacity.get(vp_id, 0))
        overload_total += excess * overload_weight_value

    total = assignment_total + overload_total

    return pd.DataFrame([
        {
            "Component": "Assignment Cost",
            "Value": assignment_total,
            "Percent of Total": (assignment_total / total) * 100 if total != 0 else 0
        },
        {
            "Component": "Overload Penalty",
            "Value": overload_total,
            "Percent of Total": (overload_total / total) * 100 if total != 0 else 0
        },
        {
            "Component": "Total Objective",
            "Value": total,
            "Percent of Total": 100
        }
    ])


def summarize_sensitivity_results(sensitivity_results):
    if sensitivity_results is None or sensitivity_results.empty:
        return "No sensitivity results available."

    best_objective_row = sensitivity_results.loc[sensitivity_results["optimized_score"].idxmin()]
    fewest_violations_row = sensitivity_results.loc[sensitivity_results["violations"].idxmin()]

    summary = f"""
Sensitivity analysis shows that optimized objective values ranged from {sensitivity_results['optimized_score'].min():.2f} to {sensitivity_results['optimized_score'].max():.2f}.
The lowest optimized objective value was {best_objective_row['optimized_score']:.2f}, produced by the configuration with distance weight {best_objective_row['distance_weight']} and complexity weight {best_objective_row['complexity_weight']}.
The fewest capacity violations were {int(fewest_violations_row['violations'])}, produced by the configuration with reassignment penalty {fewest_violations_row['reassignment_penalty']} and service-line penalty {fewest_violations_row['sl_penalty']}.
"""
    return summary.strip()


def generate_explanation_summary(comparison_results):
    obj = comparison_results["objective_comparison"]
    vio = comparison_results["violation_comparison"]
    num_reassignments = len(comparison_results["reassignments"])

    summary = f"""
The optimized system produced an objective value of {obj['new_objective']:.2f}, compared to a base value of {obj['base_objective']:.2f}.
This corresponds to an objective delta of {obj['objective_delta']:.2f} and a percent improvement of {obj['percent_improvement']:.2f}%.
Constraint violations changed from {vio['base_violations']} in the base state to {vio['new_violations']} in the optimized state.
A total of {num_reassignments} reassignment changes were made relative to the base structure.
"""
    return summary.strip()


def run_interpretability_module(
    optimized_sol,
    assignment_costs,
    optimized_workload,
    vp_capacity,
    vp_base_workload,
    comparison_results,
    sensitivity_results=None,
    overload_weight_value=2.0
):
    decision_table = build_decision_table(optimized_sol, assignment_costs)
    overload_table = summarize_overload(optimized_workload, vp_capacity)
    decomposition_table = objective_decomposition(
        optimized_sol,
        assignment_costs,
        vp_capacity,
        vp_base_workload,
        overload_weight_value
    )
    explanation_summary = generate_explanation_summary(comparison_results)
    sensitivity_summary = summarize_sensitivity_results(sensitivity_results) if sensitivity_results is not None else None

    return {
        "decision_table": decision_table,
        "overload_table": overload_table,
        "objective_decomposition": decomposition_table,
        "explanation_summary": explanation_summary,
        "sensitivity_summary": sensitivity_summary
    }


In [ ]:
interpretability_outputs = run_interpretability_module(
    optimized_sol=optimized_sol,
    assignment_costs=assignment_costs,
    optimized_workload=optimized_workload,
    vp_capacity=vp_capacity,
    vp_base_workload=vp_base_workload,
    comparison_results=comparison_results,
    sensitivity_results=sensitivity_results,
    overload_weight_value=2.0
)

interpretability_outputs["decision_table"].head()


In [ ]:
interpretability_outputs["overload_table"].head(10)


In [ ]:
interpretability_outputs["objective_decomposition"]


In [ ]:
def build_organizational_network(all_entities, sol, vp_sls=None, region_col=None):
    G = nx.Graph()

    for _, row in all_entities.iterrows():
        facility_id = row["Facility ID"]
        sl = row.get("Service Line", "Unknown")
        record_type = row.get("record_type", "facility")
        assigned_vp = sol.get(facility_id)

        G.add_node(facility_id, node_type=record_type, label=str(facility_id))

        service_node = f"Service::{sl}"
        G.add_node(service_node, node_type="sl", label=sl)
        G.add_edge(facility_id, service_node, relationship="has_service")

        if assigned_vp is not None:
            G.add_node(assigned_vp, node_type="leader", label=assigned_vp)
            G.add_edge(assigned_vp, facility_id, relationship="assigned_to")

        if region_col is not None and region_col in row.index:
            region_value = row[region_col]

            if pd.notna(region_value):
                region_node = f"Region::{region_value}"
                G.add_node(region_node, node_type="region", label=str(region_value))
                G.add_edge(facility_id, region_node, relationship="located_in")

    if vp_sls is not None:
        for vp_id, services in vp_sls.items():
            G.add_node(vp_id, node_type="leader", label=vp_id)

            for service in services:
                service_node = f"Service::{service}"
                G.add_node(service_node, node_type="sl", label=service)
                G.add_edge(vp_id, service_node, relationship="covers_service")

    return G


def summarize_organizational_graph(G):
    node_type_counts = {}

    for _, data in G.nodes(data=True):
        node_type = data.get("node_type", "unknown")
        node_type_counts[node_type] = node_type_counts.get(node_type, 0) + 1

    edge_type_counts = {}

    for _, _, data in G.edges(data=True):
        relationship = data.get("relationship", "unknown")
        edge_type_counts[relationship] = edge_type_counts.get(relationship, 0) + 1

    return {
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "node_types": node_type_counts,
        "edge_types": edge_type_counts,
        "connected_components": nx.number_connected_components(G),
        "density": nx.density(G)
    }


base_org_graph = build_organizational_network(
    all_entities=all_entities,
    sol=current_sol,
    vp_sls=vp_sls,
    region_col=None
)

optimized_org_graph = build_organizational_network(
    all_entities=all_entities,
    sol=optimized_sol,
    vp_sls=vp_sls,
    region_col=None
)

base_org_summary = summarize_organizational_graph(base_org_graph)
optimized_org_summary = summarize_organizational_graph(optimized_org_graph)

base_org_summary, optimized_org_summary


In [ ]:
def plot_leader_network(G, leader_id, depth=2):
    if leader_id not in G:
        return
    nodes = nx.single_source_shortest_path_length(G, leader_id, cutoff=depth).keys()
    subgraph = G.subgraph(nodes)
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(subgraph, seed=42)
    node_types = nx.get_node_attributes(subgraph, 'node_type')
    leader_nodes = [n for n, t in node_types.items() if t == 'leader']
    facility_nodes = [n for n, t in node_types.items() if t in ['current', 'opportunity', 'facility']]
    service_nodes = [n for n, t in node_types.items() if t == 'sl']
    region_nodes = [n for n, t in node_types.items() if t == 'region']
    nx.draw_networkx_nodes(subgraph, pos, nodelist=leader_nodes, node_size=900)
    nx.draw_networkx_nodes(subgraph, pos, nodelist=facility_nodes, node_size=300)
    nx.draw_networkx_nodes(subgraph, pos, nodelist=service_nodes, node_size=600)
    nx.draw_networkx_nodes(subgraph, pos, nodelist=region_nodes, node_size=500)
    nx.draw_networkx_edges(subgraph, pos)
    nx.draw_networkx_labels(subgraph, pos, font_size=8)
    plt.title(f'Organizational Network View for {leader_id}')
    plt.axis('off')
    plt.show()

def plot_leader_opportunities_only(leader_id, sol, all_entities):
    G = nx.Graph()
    G.add_node(leader_id, node_type='leader')
    entity_lookup_local = all_entities.drop_duplicates(subset=['Facility ID']).set_index('Facility ID')
    for facility_id, vp_id in sol.items():
        if vp_id != leader_id or facility_id not in entity_lookup_local.index:
            continue
        entity = entity_lookup_local.loc[facility_id]
        record_type = entity.get('record_type', 'Unknown')
        if record_type == 'opportunity':
            sl = entity.get('Service Line', 'Unknown')
            service_node = f'Service::{sl}'
            G.add_node(facility_id, node_type='opportunity')
            G.add_node(service_node, node_type='sl')
            G.add_edge(leader_id, facility_id, relationship='assigned_to')
            G.add_edge(facility_id, service_node, relationship='has_service')
    plt.figure(figsize=(10, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos, nodelist=[leader_id], node_size=1000)
    opportunity_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'opportunity']
    service_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'sl']
    nx.draw_networkx_nodes(G, pos, nodelist=opportunity_nodes, node_size=450)
    nx.draw_networkx_nodes(G, pos, nodelist=service_nodes, node_size=700)
    nx.draw_networkx_edges(G, pos)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.title(f'New Opportunities Assigned to {leader_id}')
    plt.axis('off')
    plt.show()

def plot_top_cost_assignments(leader_id, sol, all_entities, assignment_costs, top_n=15):
    assigned = []
    for facility_id, vp_id in sol.items():
        if vp_id == leader_id:
            cost = assignment_costs.get(facility_id, {}).get(vp_id, None)
            if cost is not None:
                assigned.append((facility_id, cost))
    assigned = sorted(assigned, key=lambda x: x[1], reverse=True)[:top_n]
    G = nx.Graph()
    G.add_node(leader_id, node_type='leader')
    entity_lookup_local = all_entities.drop_duplicates(subset=['Facility ID']).set_index('Facility ID')
    for facility_id, cost in assigned:
        if facility_id in entity_lookup_local.index:
            sl = entity_lookup_local.loc[facility_id].get('Service Line', 'Unknown')
        else:
            sl = 'Unknown'
        service_node = f'Service::{sl}'
        G.add_node(facility_id, node_type='facility', cost=cost)
        G.add_node(service_node, node_type='sl')
        G.add_edge(leader_id, facility_id, relationship='assigned_to', weight=cost)
        G.add_edge(facility_id, service_node, relationship='has_service')
    plt.figure(figsize=(12, 7))
    pos = nx.spring_layout(G, seed=42)
    leader_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'leader']
    facility_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'facility']
    service_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'sl']
    nx.draw_networkx_nodes(G, pos, nodelist=leader_nodes, node_size=1000)
    nx.draw_networkx_nodes(G, pos, nodelist=facility_nodes, node_size=500)
    nx.draw_networkx_nodes(G, pos, nodelist=service_nodes, node_size=700)
    nx.draw_networkx_edges(G, pos)
    nx.draw_networkx_labels(G, pos, font_size=8)
    plt.title(f'Top {top_n} Highest-Cost Assignments for {leader_id}')
    plt.axis('off')
    plt.show()


In [ ]:
def build_vp_comparison_table(
    base_sol,
    optimized_sol,
    base_workload,
    optimized_workload,
    vp_capacity,
    all_entities,
    assignment_costs
):
    entity_lookup_local = all_entities.drop_duplicates(subset=["Facility ID"]).set_index("Facility ID")

    base_by_vp = {}
    optimized_by_vp = {}

    for facility_id, vp_id in base_sol.items():
        base_by_vp.setdefault(vp_id, []).append(facility_id)

    for facility_id, vp_id in optimized_sol.items():
        optimized_by_vp.setdefault(vp_id, []).append(facility_id)

    all_vps = sorted(set(base_by_vp.keys()).union(set(optimized_by_vp.keys())))
    rows = []

    for vp_id in all_vps:
        base_ids = set(base_by_vp.get(vp_id, []))
        optimized_ids = set(optimized_by_vp.get(vp_id, []))

        optimized_new_opportunities = []
        optimized_assignment_costs = []

        for facility_id in optimized_ids:
            if facility_id in entity_lookup_local.index:
                record_type = entity_lookup_local.loc[facility_id].get("record_type", "current")

                if record_type == "opportunity":
                    optimized_new_opportunities.append(facility_id)

                cost = assignment_costs.get(facility_id, {}).get(vp_id, np.nan)
                optimized_assignment_costs.append(cost)

        base_wl = base_workload.get(vp_id, 0)
        optimized_wl = optimized_workload.get(vp_id, 0)
        capacity = vp_capacity.get(vp_id, 0)

        rows.append({
            "VP ID": vp_id,
            "base Workload": base_wl,
            "Optimized Workload": optimized_wl,
            "Workload Change": optimized_wl - base_wl,
            "Capacity": capacity,
            "base Excess": max(0, base_wl - capacity),
            "Optimized Excess": max(0, optimized_wl - capacity),
            "Excess Change": max(0, optimized_wl - capacity) - max(0, base_wl - capacity),
            "base Facility Count": len(base_ids),
            "Optimized Facility Count": len(optimized_ids),
            "Facility Count Change": len(optimized_ids) - len(base_ids),
            "New Opportunities Assigned": len(optimized_new_opportunities),
            "Optimized New Opportunities": len(optimized_new_opportunities),
            "Avg Assignment Cost": np.nanmean(optimized_assignment_costs) if len(optimized_assignment_costs) > 0 else 0
        })

    return pd.DataFrame(rows)


vp_comparison_df = build_vp_comparison_table(
    base_sol=current_sol,
    optimized_sol=optimized_sol,
    base_workload=vp_base_workload,
    optimized_workload=optimized_workload,
    vp_capacity=vp_capacity,
    all_entities=all_entities,
    assignment_costs=assignment_costs
)

vp_comparison_df.head()


In [ ]:
def pick_outlier_vps(vp_comparison_df):
    selected = {}
    used = set()

    category_specs = [
        ("Most Overloaded", "Optimized Excess", False),
        ("Largest Workload Increase", "Workload Change", False),
        ("Largest Workload Decrease", "Workload Change", True),
        ("Most New Opportunities", "Optimized New Opportunities", False),
    ]

    for category_name, col, ascending in category_specs:
        sorted_df = vp_comparison_df.sort_values(by=col, ascending=ascending)

        chosen_vp = None
        for _, row in sorted_df.iterrows():
            vp = row["VP ID"]
            if vp not in used:
                chosen_vp = vp
                used.add(vp)
                break

        if chosen_vp is not None:
            selected[category_name] = chosen_vp

    return selected


def build_vp_subgraph(vp_id, sol, all_entities, max_current_facilities=8):
    G = nx.Graph()
    G.add_node(vp_id, node_type="leader")
    entity_lookup_local = all_entities.drop_duplicates(subset=["Facility ID"]).set_index("Facility ID")

    assigned_ids = [fid for fid, vp in sol.items() if vp == vp_id]
    current_facilities = []
    opportunities = []

    for fid in assigned_ids:
        if fid not in entity_lookup_local.index:
            continue

        row = entity_lookup_local.loc[fid]
        record_type = row.get("record_type", "current")
        complexity = row.get("Facility Complexity Score", 0)

        if record_type == "opportunity":
            opportunities.append((fid, complexity))
        else:
            current_facilities.append((fid, complexity))

    current_facilities = sorted(current_facilities, key=lambda x: x[1], reverse=True)[:max_current_facilities]
    selected_ids = [fid for fid, _ in current_facilities] + [fid for fid, _ in opportunities]

    for fid in selected_ids:
        row = entity_lookup_local.loc[fid]
        record_type = row.get("record_type", "current")
        sl = row.get("Service Line", "Unknown")
        service_node = f"Service::{sl}"

        G.add_node(fid, node_type=record_type)
        G.add_node(service_node, node_type="sl")
        G.add_edge(vp_id, fid, relationship="assigned_to")
        G.add_edge(fid, service_node, relationship="has_service")

    return G


def draw_vp_subgraph(ax, G, title, pos=None):
    if pos is None:
        pos = nx.spring_layout(G, seed=42)

    node_types = nx.get_node_attributes(G, "node_type")

    leader_nodes = [n for n, t in node_types.items() if t == "leader"]
    current_nodes = [n for n, t in node_types.items() if t == "current"]
    opportunity_nodes = [n for n, t in node_types.items() if t == "opportunity"]
    service_nodes = [n for n, t in node_types.items() if t == "sl"]

    nx.draw_networkx_nodes(G, pos, nodelist=leader_nodes, node_size=1000, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=current_nodes, node_size=350, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=opportunity_nodes, node_size=500, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=service_nodes, node_size=700, ax=ax)
    nx.draw_networkx_edges(G, pos, ax=ax)

    labels = {n: str(n).replace("Service::", "") for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)

    ax.set_title(title)
    ax.axis("off")


def plot_before_after_vp(vp_id, category_name, base_sol, optimized_sol, vp_comparison_df, all_entities, max_current_facilities=8):
    G_before = build_vp_subgraph(vp_id, base_sol, all_entities, max_current_facilities=max_current_facilities)
    G_after = build_vp_subgraph(vp_id, optimized_sol, all_entities, max_current_facilities=max_current_facilities)
    G_union = nx.compose(G_before, G_after)
    pos = nx.spring_layout(G_union, seed=42)

    vp_row = vp_comparison_df[vp_comparison_df["VP ID"] == vp_id].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    draw_vp_subgraph(
        axes[0],
        G_before,
        title=(
            f"{category_name}\n"
            f"{vp_id} - BEFORE\n"
            f"Workload={vp_row['base Workload']:.2f}, "
            f"Excess={vp_row['base Excess']:.2f}"
        ),
        pos=pos
    )

    draw_vp_subgraph(
        axes[1],
        G_after,
        title=(
            f"{category_name}\n"
            f"{vp_id} - AFTER\n"
            f"Workload={vp_row['Optimized Workload']:.2f}, "
            f"Excess={vp_row['Optimized Excess']:.2f}, "
            f"New Opps={int(vp_row['Optimized New Opportunities'])}"
        ),
        pos=pos
    )

    plt.tight_layout()
    plt.show()


In [ ]:
outlier_vps = pick_outlier_vps(vp_comparison_df)
outlier_vps


In [ ]:
for category_name, vp_id in outlier_vps.items():
    plot_before_after_vp(vp_id=vp_id, category_name=category_name, base_sol=current_sol, optimized_sol=optimized_sol, vp_comparison_df=vp_comparison_df, all_entities=all_entities, max_current_facilities=8)


In [ ]:
def plot_base_vs_optimized_workload(vp_comparison_df, label_top_n=5):
    df = vp_comparison_df.copy()
    plt.figure(figsize=(9, 7))
    plt.scatter(df['base Workload'], df['Optimized Workload'], s=60)
    min_val = min(df['base Workload'].min(), df['Optimized Workload'].min())
    max_val = max(df['base Workload'].max(), df['Optimized Workload'].max())
    plt.plot([min_val, max_val], [min_val, max_val], linestyle='--', label='No Change Line')
    label_df = df.reindex(df['Workload Change'].abs().sort_values(ascending=False).index).head(label_top_n)
    for _, row in label_df.iterrows():
        plt.annotate(row['VP ID'], (row['base Workload'], row['Optimized Workload']), textcoords='offset points', xytext=(5, 5), fontsize=8)
    plt.xlabel('base Workload')
    plt.ylabel('Optimized Workload')
    plt.title('base vs Optimized Workload by Leader')
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_workload_slope_chart(vp_comparison_df, top_n=10):
    df = vp_comparison_df.copy()
    df = df.reindex(df['Workload Change'].abs().sort_values(ascending=False).index).head(top_n)
    plt.figure(figsize=(10, 7))
    for _, row in df.iterrows():
        plt.plot(['base', 'Optimized'], [row['base Workload'], row['Optimized Workload']], marker='o')
        plt.text(-0.05, row['base Workload'], row['VP ID'], ha='right', va='center', fontsize=8)
        plt.text(1.05, row['Optimized Workload'], row['VP ID'], ha='left', va='center', fontsize=8)
    plt.ylabel('Workload')
    plt.title(f'Top {top_n} Leaders by Workload Change')
    plt.tight_layout()
    plt.show()

def plot_op_impact_bubble_chart(vp_comparison_df, label_top_n=5):
    df = vp_comparison_df.copy()
    bubble_size = (df['New Opportunities Assigned'] + 1) * 120
    plt.figure(figsize=(10, 7))
    plt.scatter(df['Workload Change'], df['Avg Assignment Cost'], s=bubble_size, alpha=0.6)
    plt.axvline(0, linestyle='--')
    label_df = df.sort_values(by=['New Opportunities Assigned', 'Workload Change'], ascending=False).head(label_top_n)
    for _, row in label_df.iterrows():
        plt.annotate(row['VP ID'], (row['Workload Change'], row['Avg Assignment Cost']), textcoords='offset points', xytext=(5, 5), fontsize=8)
    plt.xlabel('Workload Change')
    plt.ylabel('Avg Assignment Cost')
    plt.title('op Impact by Leader')
    plt.tight_layout()
    plt.show()

def plot_workload_capacity_for_new_opportunity_leaders(vp_comparison_df, top_n=15):
    df = vp_comparison_df.copy()
    df = df[df['New Opportunities Assigned'] > 0]
    df = df.sort_values(by='Optimized Excess', ascending=False).head(top_n)
    if df.empty:
        return
    plt.figure(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.35
    plt.bar(x - width / 2, df['base Workload'], width, label='base Workload')
    plt.bar(x + width / 2, df['Optimized Workload'], width, label='Optimized Workload')
    plt.plot(x, df['Capacity'], marker='o', label='Capacity')
    plt.xticks(x, df['VP ID'], rotation=45)
    plt.xlabel('Leader')
    plt.ylabel('Workload')
    plt.title('Workload and Capacity for Leaders Receiving New Opportunities')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_base_vs_optimized_workload(vp_comparison_df)
plot_workload_slope_chart(vp_comparison_df, top_n=10)
plot_op_impact_bubble_chart(vp_comparison_df)
plot_workload_capacity_for_new_opportunity_leaders(vp_comparison_df)


In [ ]:
final_assignment_table = build_decision_table(optimized_sol, assignment_costs)
leader_capacity_table = summarize_overload(optimized_workload, vp_capacity)

final_assignment_table.to_csv("optimized_assignments.csv", index=False)
vp_comparison_df.to_csv("vp_comparison.csv", index=False)
leader_capacity_table.to_csv("leader_capacity_summary.csv", index=False)

files.download("optimized_assignments.csv")
files.download("vp_comparison.csv")
files.download("leader_capacity_summary.csv")
